# Gold setup (`08_gold_audit_table`)

This notebook prepares what every Gold notebook needs:

1. the schemas `gold`, `gold_staging` and `audit`,
2. the audit table `github_lakehouse.audit.gold_pipeline_runs`.

It contains only `CREATE ... IF NOT EXISTS` statements. It never changes or deletes anything that already exists, so it is safe to run again.

Run it once before the first Gold notebook. In the Job it is the first Gold task.

Contract: `docs/gold_contracts.md`, sections 4 and 7.

## 1. Create the schemas

| Schema | What it holds |
|---|---|
| `gold` | The six Gold tables. Nothing else. |
| `gold_staging` | The staging copies. A Gold notebook writes its result here first, checks it, and only then publishes it to `gold`. These tables are overwritten on every run and are not meant for reading. |
| `audit` | The run logs. It already holds the Silver audit table. |

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS github_lakehouse.gold;

CREATE SCHEMA IF NOT EXISTS github_lakehouse.gold_staging;

CREATE SCHEMA IF NOT EXISTS github_lakehouse.audit;

## 2. Create the Gold audit table

Every run of a Gold notebook adds exactly one row to this table. The row says what the run did and whether it was successful.

| Column | Meaning |
|---|---|
| `audit_id` | Unique ID of this audit row |
| `run_id` | The Job run ID, or `manual-...` for a run by hand |
| `pipeline_name` | For example `gold_repository_issue_flow` |
| `gold_table` | Full name of the table that was written |
| `processing_scope` | `SINGLE_DATE` or `ALL_DATES` |
| `source_date` | The `source_date` parameter. NULL for all dates |
| `affected_dates` | The days this run rebuilt |
| `run_started_at`, `run_finished_at` | Start and end of the run (UTC) |
| `status` | `SUCCEEDED`, `NO_INPUT` or `FAILED` |
| `error_message` | NULL unless the run failed |
| `source_rows_read` | Rows used, per source table |
| `rows_excluded` | Rows left out, per reason |
| `gold_rows_published` | Rows written to the Gold table |
| `gold_table_version` | Delta version of the Gold table after the run |
| `checks_passed` | True when every quality gate passed |
| `failed_checks` | Names of the failed gates, separated by `; `. NULL when all passed |
| `quality_metrics` | Counts that belong to one table only, for example the G5 release events left out per reason |

The three statuses:

- `SUCCEEDED`: the quality gates passed and the result was published.
- `NO_INPUT`: there were no rows for this run. The Gold table is unchanged and the Job task does not fail.
- `FAILED`: a quality gate or a write failed. Nothing was published. The notebook writes the audit row first and then stops with an error, so the Job task fails too.

In [0]:
%sql
CREATE TABLE IF NOT EXISTS github_lakehouse.audit.gold_pipeline_runs (
    -- Which run wrote this row
    audit_id             STRING,
    run_id               STRING,
    pipeline_name        STRING,
    gold_table           STRING,

    -- What the run covered
    processing_scope     STRING,
    source_date          DATE,
    affected_dates       ARRAY<DATE>,

    -- When it ran and how it ended
    run_started_at       TIMESTAMP,
    run_finished_at      TIMESTAMP,
    status               STRING,
    error_message        STRING,

    -- What it read and what it left out
    source_rows_read     MAP<STRING, BIGINT>,
    rows_excluded        MAP<STRING, BIGINT>,

    -- What it wrote
    gold_rows_published  BIGINT,
    gold_table_version   BIGINT,

    -- The quality gates
    checks_passed        BOOLEAN,
    failed_checks        STRING,

    -- Counts that belong to one table only
    quality_metrics      MAP<STRING, BIGINT>
)
USING DELTA;

## 3. Look at the result

The table should show the 18 columns listed above.

In [0]:
%sql
DESCRIBE TABLE github_lakehouse.audit.gold_pipeline_runs;